# Scaling the causal ConvLSTM

This experiment scales data, temporal context, and model capacity together. Base, Large, and XL candidates are selected strictly by validation loss; the held-out test split is reported only for final comparison.

In [ ]:
from pathlib import Path
import subprocess, sys, tempfile

def project_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    work = Path('/kaggle/working')
    if work.exists():
        candidates += [p.parent for p in work.glob('*/pyproject.toml')]
    for candidate in candidates:
        marker = candidate / 'pyproject.toml'
        if marker.exists() and 'hay-single-compartment' in marker.read_text():
            return candidate
    destination = Path(tempfile.mkdtemp(prefix='hay_scaling_', dir='/kaggle/working'))
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/Zagred47/LearningSingleCompartiment.git', str(destination)])
    return destination

ROOT = project_root()
SRC = ROOT / 'src'
assert (SRC / 'hay_single_compartment').is_dir(), f'Package source missing: {SRC}'
sys.path.insert(0, str(SRC))
print('Project:', ROOT)
print('Source:', SRC)

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from hay_single_compartment import INPUT_NAMES, STATE_NAMES, SimulationConfig, generate_dataset, validate_dataset
from hay_single_compartment.dataset import Normalization
from hay_single_compartment.models import build_model
from hay_single_compartment.training import rollout_trajectory, train_model

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT = Path('/kaggle/working/hay_convlstm_scaling') if Path('/kaggle').exists() else ROOT / 'artifacts' / 'scaling'
OUTPUT.mkdir(parents=True, exist_ok=True)
DATASET = OUTPUT / 'single_compartment_scale.h5'
print('Device:', DEVICE)

## 1. Scale the data before the network

The capacity sweep uses 24 train, 4 validation, and 4 test trajectories of 500 ms. Seeds and complete trajectories remain isolated across splits.

In [ ]:
config = SimulationConfig(
    duration_ms=500.0, warmup_ms=150.0, seed=31415,
    train_trajectories=24, validation_trajectories=4, test_trajectories=4,
)
report = generate_dataset(DATASET, config) if not DATASET.exists() else validate_dataset(DATASET)
report

## 2. Capacity sweep

All candidates use causal convolutions and LSTM memory. Larger models receive longer contexts, smaller batches, and lower learning rates. AMP is enabled automatically on CUDA.

In [ ]:
EXPERIMENTS = [
    dict(run_name='conv_base', hidden_dim=96, layers=2, width_multiplier=2, sequence_length=64, batch_size=64, learning_rate=8e-4, epochs=30),
    dict(run_name='conv_large', hidden_dim=128, layers=3, width_multiplier=2, sequence_length=128, batch_size=32, learning_rate=6e-4, epochs=40),
    dict(run_name='conv_xl', hidden_dim=160, layers=3, width_multiplier=2, sequence_length=128, batch_size=24, learning_rate=5e-4, epochs=50),
]

for experiment in EXPERIMENTS:
    probe = build_model(
        'conv_lstm', len(STATE_NAMES) + len(INPUT_NAMES), len(STATE_NAMES),
        hidden_dim=experiment['hidden_dim'], layers=experiment['layers'],
        width_multiplier=experiment['width_multiplier'],
    )
    experiment['parameters'] = sum(p.numel() for p in probe.parameters())
pd.DataFrame(EXPERIMENTS)[['run_name', 'parameters', 'sequence_length', 'epochs', 'learning_rate']]

In [ ]:
reports = []
for experiment in EXPERIMENTS:
    print('\nTraining', experiment['run_name'], f"({experiment['parameters']:,} parameters)")
    reports.append(train_model(
        DATASET, OUTPUT / 'models', 'conv_lstm',
        run_name=experiment['run_name'], epochs=experiment['epochs'],
        sequence_length=experiment['sequence_length'], stride=32,
        batch_size=experiment['batch_size'], hidden_dim=experiment['hidden_dim'],
        layers=experiment['layers'], width_multiplier=experiment['width_multiplier'],
        learning_rate=experiment['learning_rate'], patience=8, minimum_epochs=18,
        device=DEVICE, seed=31415, use_amp=True,
    ))
print('Sweep complete')

In [ ]:
comparison = pd.DataFrame([{
    'run': report['run_name'], 'parameters': report['parameters'],
    'epochs_trained': report['epochs_trained'],
    'validation_loss': report['best_validation_loss'],
    'test_voltage_rmse_mV': report['test']['voltage_rmse_mv'],
    'test_normalized_rmse': report['test']['mean_normalized_rmse'],
    'persistence_voltage_rmse_mV': report['test']['persistence_voltage_rmse_mv'],
} for report in reports]).sort_values('validation_loss')
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for report in reports:
    history = pd.DataFrame(report['history'])
    axes[0].plot(history.epoch, history.validation_loss, label=report['run_name'])
    axes[1].plot(history.epoch, history.learning_rate, label=report['run_name'])
axes[0].set(title='Validation', xlabel='epoch', ylabel='weighted loss', yscale='log')
axes[1].set(title='Cosine schedule', xlabel='epoch', ylabel='learning rate')
for axis in axes: axis.grid(alpha=.2); axis.legend()
plt.tight_layout()

## 3. Roll out the validation-selected winner

Selection uses validation loss only. The following 200 ms rollout measures compounding error on the held-out test trajectory.

In [ ]:
winner_name = comparison.iloc[0]['run']
checkpoint = torch.load(OUTPUT / 'models' / f'{winner_name}.pt', map_location=DEVICE, weights_only=False)
winner = build_model(
    checkpoint['architecture'], len(STATE_NAMES) + len(INPUT_NAMES), len(STATE_NAMES),
    **checkpoint['model_kwargs'],
).to(DEVICE)
winner.load_state_dict(checkpoint['model_state'])
normalization = Normalization.from_dict(checkpoint['normalization'])
with h5py.File(DATASET, 'r') as h5:
    truth = h5['test/states'][0, :2001]
    future_inputs = h5['test/inputs'][0, :2000]
prediction = rollout_trajectory(winner, truth[0], future_inputs, normalization, DEVICE)
horizons = (1, 5, 10, 25, 50, 100, 150, 200)
rollout_scores = pd.DataFrame([{
    'horizon_ms': horizon,
    'voltage_rmse_mV': float(np.sqrt(np.mean((prediction[:int(horizon/config.dt_ms)+1, 0] - truth[:int(horizon/config.dt_ms)+1, 0])**2))),
} for horizon in horizons])
print('Validation-selected winner:', winner_name)
rollout_scores

In [ ]:
time_ms = np.arange(len(truth)) * config.dt_ms
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(time_ms, truth[:, 0], label='teacher', lw=1)
axes[0].plot(time_ms, prediction[:, 0], label=winner_name, lw=1, alpha=.8)
axes[0].set_ylabel('V (mV)'); axes[0].legend()
axes[1].plot(time_ms, truth[:, 1] * 1e3, label='teacher')
axes[1].plot(time_ms, prediction[:, 1] * 1e3, label=winner_name, alpha=.8)
axes[1].set(xlabel='time (ms)', ylabel='Ca i (uM)'); axes[1].legend()
plt.tight_layout()

## Reading the result

A larger model is useful only if validation and held-out rollout improve together. If XL loses to Large, increase trajectory diversity before adding parameters. The generated HDF5, checkpoints, metrics, and figures remain in `/kaggle/working/hay_convlstm_scaling`.